# Libraries

In [1]:
# !pip install -q vllm
# !pip install -q -U llmcompressor
# !pip install -q compressed-tensors

In [2]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed

# Deep learning framework
import torch

/home/toqa/vllm-env/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/toqa/vllm-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Global Variables

In [ ]:
# User & Repository Configuration
USER_NAME = ""
EMAIL     = ""
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
QWEN  = False
MODEL = "Qwen3-4B" if QWEN else "Llama-3.2-3B-Instruct"

# Authentication Token
GIT_TOKEN = ""
HF_TOKEN = ""

# Model & Tokenizer Configuration
MODEL_ID     = "EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit"
TOKENIZER_ID = "meta-llama/Llama-3.2-3B-Instruct"

MODEL_NAME           = "Llama-3.2-3B-Instruct"
MODEL_NAME_IN_REPO   = "Llama-3.2-3B-Instruct-AWQ"
COMPRESSION_METHOD   = "AWQ"
MODEL_TYPE           = "Quantization"
# SPARSITY = 25  # Uncomment if sparsity is applied


# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]


# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42


# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [6]:
login(HF_TOKEN)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Prompt Preprocessing

**DummyPrompt Tokenization**

In [7]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

**RealPrompt Tokenization**

In [8]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [9]:
llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.80,
    trust_remote_code=True,
    attention_backend = "FLASH_ATTN"
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 04-20 13:08:30 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 256, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'attention_backend': 'FLASH_ATTN', 'model': 'EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit'}
INFO 04-20 13:08:31 [model.py:549] Resolved architecture: LlamaForCausalLM
WARNING 04-20 13:08:31 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 04-20 13:08:31 [model.py:1678] Using max model len 256
INFO 04-20 13:08:35 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 13:08:35 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 13:08:36 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: WSL is detected and NVML is not compatible with fork


/home/toqa/vllm-env/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


(EngineCore pid=24367) INFO 04-20 13:08:49 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit', speculative_config=None, tokenizer='EdgeCompress01/Llama-3.2-3B-Instruct-AWQ-4bit', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=256, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=compressed-tensors, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_fo

(EngineCore pid=24367) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=24367) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=24367) INFO 04-20 13:09:00 [weight_utils.py:625] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.49s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.49s/it]
(EngineCore pid=24367) 


(EngineCore pid=24367) INFO 04-20 13:09:02 [default_loader.py:384] Loading weights took 2.88 seconds
(EngineCore pid=24367) INFO 04-20 13:09:02 [gpu_model_runner.py:4820] Model loading took 2.13 GiB memory and 7.654733 seconds
(EngineCore pid=24367) INFO 04-20 13:09:06 [backends.py:1051] Using cache directory: /home/toqa/.cache/vllm/torch_compile_cache/a0b034876d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=24367) INFO 04-20 13:09:06 [backends.py:1111] Dynamo bytecode transform time: 3.05 s
(EngineCore pid=24367) INFO 04-20 13:09:07 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.008 s
(EngineCore pid=24367) INFO 04-20 13:09:07 [decorators.py:303] Directly load AOT compilation from path /home/toqa/.cache/vllm/torch_compile_cache/torch_aot_compile/43c83a7c52b213d18f4e7e8e84d89e2a09b4b750010b902f20afddc75ad8d431/rank_0_0/model
(EngineCore pid=24367) INFO 04-20 13:09:07 [monitor.py:48] torch.compile took 4.62 s in tota

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:05<00:00,  9.55it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 12.87it/s]


(EngineCore pid=24367) INFO 04-20 13:09:21 [gpu_model_runner.py:6046] Graph capturing finished in 9 secs, took 0.25 GiB
(EngineCore pid=24367) INFO 04-20 13:09:21 [gpu_worker.py:597] CUDA graph pool memory: 0.25 GiB (actual), 0.09 GiB (estimated), difference: 0.15 GiB (62.0%).
(EngineCore pid=24367) INFO 04-20 13:09:21 [core.py:283] init engine (profile, create kv cache, warmup model) took 18.97 seconds
(EngineCore pid=24367) INFO 04-20 13:09:23 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 13:09:23 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
vLLM Initialized Successfully!


**Warm up**

In [10]:
_ = llm.generate(
    prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
    sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
    use_tqdm=True
)

# CLEAN UP & SYNCHRONIZE
gc.collect()
torch.cuda.empty_cache()

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s, est. speed input: 57.23 toks/s, output: 66.55 toks/s]


# Inference

**TTFT**

In [11]:
start_time_ttft = time.perf_counter()

_ = llm.generate(
    prompts=[{"prompt_token_ids": prompt_token_ids}], 
    sampling_params=ttft_sampling_params, 
    use_tqdm=False
)

end_time_ttft = time.perf_counter()


# CLEAN UP & SYNCHRONIZE AGAIN
gc.collect()
torch.cuda.empty_cache()

**Full Evaluations**

In [12]:
start_time = time.perf_counter()

outputs = llm.generate(
    prompts=[{"prompt_token_ids": prompt_token_ids}], 
    sampling_params=sampling_params, 
    use_tqdm=False
)

end_time = time.perf_counter()

# Results

**Calculations**

In [13]:
final_output_text = outputs[0].outputs[0].text

input_tokens = len(prompt_token_ids)

total_generated_tokens = MAX_GENERATION_TOKENS

ttft_seconds = end_time_ttft - start_time_ttft

latency = end_time - start_time

tps = total_generated_tokens / latency

**Writing in json file**

In [14]:
benchmark_results = {
    "model_name": MODEL_NAME,
    "compression_method": COMPRESSION_METHOD,
    "model_type": MODEL_TYPE,
    #"sparsity": SPARSITY,
    "prompt_text": prompt_text,
    "generated_text": final_output_text,
    "input_token_count": input_tokens,
    "generated_token_count": total_generated_tokens,
    "time_to_first_token_sec": ttft_seconds,
    "end_to_end_latency_sec": latency,
    "throughput_tokens_per_sec": tps,
}


# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [15]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/{MODEL}/{MODEL_NAME_IN_REPO}"
source_file = OUTPUT_FILE 
remote_url = f"https://{GIT_TOKEN}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Add local inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /home/toqa/temp_git_repo


Cloning into '/home/toqa/temp_git_repo'...


[main ad51206] Add local inference evaluation results to Evaluations/InferenceEvaluations/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-AWQ
 1 file changed, 4 insertions(+), 4 deletions(-)
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-AWQ'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   80ecefa..ad51206  main -> main
